# Parameter sweep recovery

Load one completed parameter-recovery sweep, remove failed MCMC fits from aggregate recovery summaries, and inspect per-parameter SBC diagnostics interactively.

The notebook keeps its analysis helpers local so it can be run from the project root or from `notebooks/`. Select the artifact directory below, then run all cells in order.


In [ ]:
from collections.abc import Sequence
from pathlib import Path
import os
import sys

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib")

import arviz as az
from arviz_stats.ecdf_utils import ecdf_pit
from IPython.display import Markdown, display
import ipywidgets as widgets
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import numpy as np
import pandas as pd
import xarray as xr

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from coupling_sources.core.priors import CouplingPriors
from coupling_sources.utils.artifacts import Artifacts

A4_PORTRAIT_INCHES = (8.27, 11.69)
A4_LANDSCAPE_INCHES = (11.69, 8.27)
DIAGNOSTICS_FIGURE_SIZE = A4_PORTRAIT_INCHES
R_HAT_THRESHOLD = 1.05
MIN_ESS = 100.0
CONFIGURATION_ROW_HEIGHT_INCHES = 0.18
CONFIGURATION_COLUMNS = (
    "n_participants",
    "n_trials_full",
    "n_trials_partial",
    "sigma",
    "lam",
)

## Load sweep artifacts

Missing posterior artifacts are reported and skipped; every downstream summary is available-case. `stimuli_seed` is retained as provenance but deliberately excluded from stimulus configuration identity because recovery jobs can use randomized stimulus seeds.


In [ ]:
def load_parameter_sweep(
    run_dir: Path | str,
    *,
    parameters: Sequence[str] | None = None,
    configuration_columns: Sequence[str] | None = None,
) -> xr.Dataset:
    run_dir = Path(run_dir)
    parameter_names = tuple(parameters or CouplingPriors.parameter_sites())

    grid_path = run_dir / "job_grid.csv"
    job_grid = pd.read_csv(grid_path, dtype=str, keep_default_na=False)
    if job_grid.empty:
        raise ValueError(f"empty job grid: {grid_path}")

    posterior_paths = [
        path if path.is_absolute() else run_dir / path
        for path in job_grid["posterior_path"].map(Path)
    ]
    complete = np.array([path.is_file() for path in posterior_paths])
    missing_job_ids = job_grid.loc[~complete, "job_id"].tolist()
    if not complete.any():
        raise ValueError(f"no completed posterior jobs in {run_dir}")
    if missing_job_ids:
        preview = ", ".join(missing_job_ids[:10])
        suffix = " ..." if len(missing_job_ids) > 10 else ""
        print(
            f"Skipping {len(missing_job_ids)} job(s) without posterior artifacts: "
            f"{preview}{suffix}"
        )

    job_grid = job_grid.loc[complete].reset_index(drop=True)
    posterior_paths = [
        path
        for path, is_complete in zip(posterior_paths, complete, strict=True)
        if is_complete
    ]
    stimulus_columns = tuple(
        column
        for column in job_grid.columns
        if column.startswith("stimuli_") and column != "stimuli_path"
    )
    if configuration_columns is None:
        configuration_columns = tuple(
            column for column in stimulus_columns if column != "stimuli_seed"
        )
    else:
        configuration_columns = tuple(configuration_columns)
        unknown = set(configuration_columns).difference(stimulus_columns)
        if unknown:
            raise ValueError(
                f"unknown stimulus configuration columns: {sorted(unknown)}"
            )

    configuration_index = pd.MultiIndex.from_frame(
        job_grid[list(configuration_columns)]
    )
    stimulus_config = pd.factorize(configuration_index, sort=False)[0] + 1
    label_columns = tuple(
        column for column in configuration_columns if job_grid[column].nunique() > 1
    )
    stimulus_label = np.array(
        [
            ", ".join(
                f"{column.removeprefix('stimuli_')}={row[column]}"
                for column in label_columns
            )
            or f"stimulus {config}"
            for row, config in zip(
                job_grid.to_dict("records"), stimulus_config, strict=True
            )
        ]
    )

    posterior_artifacts = [
        Artifacts.read_dist_samples(path) for path in posterior_paths
    ]
    diagnostics = [Artifacts.read_mcmc_diagnostics(path) for path in posterior_paths]
    raw_prior = Artifacts.read_dist_samples(run_dir / "prior.npz")

    posterior = np.stack(
        [
            np.stack(
                [
                    np.asarray(samples[parameter], dtype=float).reshape(-1)
                    for parameter in parameter_names
                ]
            )
            for samples in posterior_artifacts
        ]
    )
    posterior = np.where(np.isfinite(posterior), posterior, np.nan)
    prior = np.stack(
        [
            np.asarray(raw_prior[parameter], dtype=float).reshape(-1)
            for parameter in parameter_names
        ]
    )
    prior = np.where(np.isfinite(prior), prior, np.nan)
    truth_columns = [f"simulation_{parameter}" for parameter in parameter_names]
    truth = job_grid[truth_columns].to_numpy(dtype=float)

    return xr.Dataset(
        data_vars={
            "posterior": (("job_id", "parameter", "posterior_draw"), posterior),
            "prior": (("parameter", "prior_draw"), prior),
            "truth": (("job_id", "parameter"), truth),
            "r_hat": (
                ("job_id", "parameter"),
                np.array(
                    [
                        [item["r_hat"][parameter] for parameter in parameter_names]
                        for item in diagnostics
                    ],
                    dtype=float,
                ),
            ),
            "ess": (
                ("job_id", "parameter"),
                np.array(
                    [
                        [item["ess"][parameter] for parameter in parameter_names]
                        for item in diagnostics
                    ],
                    dtype=float,
                ),
            ),
            "has_divergences": (
                ("job_id",),
                np.array([bool(item["has_divergences"]) for item in diagnostics]),
            ),
            "n_divergences": (
                ("job_id",),
                np.array([item["n_divergences"] or 0 for item in diagnostics]),
            ),
        },
        coords={
            "job_id": job_grid["job_id"].to_numpy(),
            "parameter": list(parameter_names),
            "posterior_draw": np.arange(1, posterior.shape[-1] + 1),
            "prior_draw": np.arange(1, prior.shape[-1] + 1),
            "stimulus_config": ("job_id", stimulus_config),
            "stimulus_label": ("job_id", stimulus_label),
            **{
                column: ("job_id", job_grid[column].to_numpy())
                for column in stimulus_columns
            },
            **{
                column: ("job_id", truth[:, index])
                for index, column in enumerate(truth_columns)
            },
        },
        attrs={
            "run_dir": str(run_dir),
            "total_jobs": len(complete),
            "missing_jobs": len(missing_job_ids),
            "configuration_columns": list(configuration_columns),
        },
    )

## Select and load a completed run


In [ ]:
RUN_DIR = PROJECT_ROOT / "artifacts" / "parameters_sweep_adam_2026_07_20_16_39_22"
RUN_DIR

In [ ]:
sweep = load_parameter_sweep(RUN_DIR)
posterior_summary = xr.Dataset(
    {
        "median": sweep["posterior"].median("posterior_draw", skipna=True),
        "truth": sweep["truth"],
    }
)

display(
    pd.Series(
        {
            "completed jobs": sweep.sizes["job_id"],
            "missing jobs": sweep.attrs["missing_jobs"],
            "stimulus configurations": np.unique(sweep["stimulus_config"]).size,
            "parameters": sweep.sizes["parameter"],
            "posterior draws per job": sweep.sizes["posterior_draw"],
            "jobs with divergences": int(sweep["has_divergences"].sum()),
        },
        name="Sweep summary",
    )
)

## MCMC quality

A parameter-job is considered invalid when the job diverged, its parameter-specific `R-hat` exceeds 1.05, or its effective sample size is below 100. Aggregate recovery plots below exclude these observations. Interactive diagnostics retain every job and mark MCMC issues with a small black triangle at the true value.


In [ ]:
def stimulus_view(sweep: xr.Dataset, stimulus_config: int | str) -> xr.Dataset:
    coordinate = (
        "stimulus_label" if isinstance(stimulus_config, str) else "stimulus_config"
    )
    job_ids = sweep["job_id"].where(sweep[coordinate] == stimulus_config, drop=True)
    if job_ids.size == 0:
        raise KeyError(f"unknown stimulus config: {stimulus_config}")
    return sweep.sel(job_id=job_ids)


def mcmc_issue(
    view: xr.Dataset,
    parameter: str,
    *,
    r_hat_threshold: float = R_HAT_THRESHOLD,
    min_ess: float = MIN_ESS,
) -> xr.DataArray:
    r_hat = view["r_hat"].sel(parameter=parameter, drop=True)
    ess = view["ess"].sel(parameter=parameter, drop=True)
    return (
        view["has_divergences"]
        | ~np.isfinite(r_hat)
        | ~np.isfinite(ess)
        | (r_hat > r_hat_threshold)
        | (ess < min_ess)
    )


def parameter_sbc(
    view: xr.Dataset,
    *,
    parameter: str,
) -> xr.Dataset:
    posterior = view["posterior"].sel(parameter=parameter)
    truth = view["truth"].sel(parameter=parameter)
    finite = posterior.notnull()
    n_draws = finite.sum("posterior_draw")
    rank = (posterior < truth).sum("posterior_draw") + 0.5 * (posterior == truth).sum(
        "posterior_draw"
    )
    pit = (rank + 0.5) / (n_draws + 1)
    sbc = xr.Dataset(
        {
            "truth": truth,
            "n_draws": n_draws,
            "rank": rank,
            "pit": pit,
            "coverage_score": 2 * abs(pit - 0.5),
        }
    )
    return sbc.where((sbc["n_draws"] > 0) & sbc["truth"].notnull(), drop=True)

## Recovery error by configuration

Diamonds show mean absolute error among valid fits. Horizontal lines show the 10th-90th percentile range. MCMC failure rates are reported separately by parameter and participant count.

Set `CONFIGURATION_ROW_HEIGHT_INCHES` in the setup cell to control vertical spacing. Smaller values produce a shorter, denser figure.


In [ ]:
def numeric_stimulus_coordinate(sweep: xr.Dataset, name: str) -> np.ndarray:
    coordinate = sweep[f"stimuli_{name}"].to_numpy()
    try:
        return coordinate.astype(float)
    except ValueError as error:
        raise ValueError(f"stimulus coordinate {name!r} must be numeric") from error


def build_recovery_error_data(
    sweep: xr.Dataset,
    posterior_summary: xr.Dataset,
) -> pd.DataFrame:
    parameter_names = [str(value) for value in sweep["parameter"].to_numpy()]
    configuration = {
        name: numeric_stimulus_coordinate(sweep, name) for name in CONFIGURATION_COLUMNS
    }
    frames = []
    for parameter in parameter_names:
        summary = posterior_summary.sel(parameter=parameter)
        frames.append(
            pd.DataFrame(
                {
                    "job_id": sweep["job_id"].to_numpy().astype(str),
                    "parameter": parameter,
                    **configuration,
                    "absolute_error": np.abs(
                        summary["median"].to_numpy() - summary["truth"].to_numpy()
                    ),
                    "mcmc_issue": mcmc_issue(sweep, parameter).to_numpy().astype(bool),
                }
            )
        )

    data = pd.concat(frames, ignore_index=True)
    data = data[np.isfinite(data["absolute_error"])].reset_index(drop=True)
    for name in CONFIGURATION_COLUMNS:
        levels = np.sort(data[name].unique())
        data[name] = pd.Categorical(data[name], categories=levels, ordered=True)
    data["parameter"] = pd.Categorical(
        data["parameter"], categories=parameter_names, ordered=True
    )
    return data


def summarize_mcmc_quality(data: pd.DataFrame) -> pd.DataFrame:
    quality = (
        data.groupby(["parameter", "n_participants"], observed=True)
        .agg(
            jobs=("job_id", "size"),
            failures=("mcmc_issue", "sum"),
            failure_rate=("mcmc_issue", "mean"),
        )
        .reset_index()
    )
    return quality.sort_values(["parameter", "n_participants"]).reset_index(drop=True)


def summarize_configuration_errors(data: pd.DataFrame) -> pd.DataFrame:
    return (
        data.groupby(["parameter", *CONFIGURATION_COLUMNS], observed=True)
        .agg(
            mean_absolute_error=("absolute_error", "mean"),
            q10_absolute_error=("absolute_error", lambda values: values.quantile(0.10)),
            q90_absolute_error=("absolute_error", lambda values: values.quantile(0.90)),
            n_jobs=("job_id", "size"),
        )
        .reset_index()
        .sort_values(["parameter", *CONFIGURATION_COLUMNS])
        .reset_index(drop=True)
    )

In [ ]:
def plot_configuration_error_ranges(
    summary: pd.DataFrame,
) -> dict[str, plt.Figure]:
    figures = {}
    for parameter, parameter_summary in summary.groupby("parameter", observed=True):
        view = parameter_summary.sort_values(list(CONFIGURATION_COLUMNS)).reset_index(
            drop=True
        )
        y = np.arange(len(view) - 1, -1, -1, dtype=float)
        paths = [
            tuple(float(row[name]) for name in CONFIGURATION_COLUMNS)
            for _, row in view.iterrows()
        ]
        fig, (tree_ax, ranking_ax) = plt.subplots(
            1,
            2,
            figsize=(16, max(8, 1.5 + CONFIGURATION_ROW_HEIGHT_INCHES * len(view))),
            sharey=True,
            gridspec_kw={"width_ratios": [1.9, 1.15], "wspace": 0.0},
            constrained_layout=True,
        )
        fig.get_layout_engine().set(w_pad=0.0, wspace=0.0)
        lower = view["q10_absolute_error"].to_numpy()
        upper = view["q90_absolute_error"].to_numpy()
        ranking_ax.hlines(y, lower, upper, color="#527988", linewidth=1.4, zorder=2)
        endpoints = np.concatenate((lower, upper))
        endpoint_y = np.concatenate((y, y))
        ranking_ax.vlines(
            endpoints,
            endpoint_y - 0.12,
            endpoint_y + 0.12,
            color="#527988",
            linewidth=0.8,
            zorder=2,
        )
        ranking_ax.scatter(
            view["mean_absolute_error"],
            y,
            marker="D",
            color="#e76f2e",
            edgecolor="white",
            linewidth=0.45,
            s=22,
            zorder=3,
        )
        ranking_ax.set_yticks(y)
        ranking_ax.set_yticklabels([])
        ranking_ax.tick_params(axis="y", length=0)
        ranking_ax.set_xlabel("absolute error |posterior median - truth|")
        ranking_ax.set_title(
            f"{parameter}: mean and 10th-90th percentile", loc="left", pad=16
        )
        ranking_ax.set_xlim(left=0)
        ranking_ax.set_ylim(-1, len(view))
        ranking_ax.grid(axis="x", color="#d8d8d8", linewidth=0.8, alpha=0.65)
        ranking_ax.grid(axis="y", color="#e8ecee", linewidth=0.45, alpha=0.7)
        ranking_ax.spines[["top", "right", "left"]].set_visible(False)
        ranking_ax.legend(
            handles=[
                Line2D(
                    [],
                    [],
                    color="#527988",
                    linewidth=1.4,
                    label="10th-90th percentile",
                ),
                Line2D(
                    [],
                    [],
                    marker="D",
                    linestyle="None",
                    color="#e76f2e",
                    markersize=4,
                    label="mean absolute error",
                ),
            ],
            loc="lower right",
            bbox_to_anchor=(1.0, 1.0),
            borderaxespad=0.0,
            ncols=2,
            frameon=False,
            fontsize=7,
        )

        positions: dict[tuple[float, ...], list[float]] = {}
        for path, y_position in zip(paths, y, strict=True):
            for depth in range(len(path) + 1):
                positions.setdefault(path[:depth], []).append(y_position)
        centers = {path: float(np.mean(values)) for path, values in positions.items()}

        for depth in range(1, len(CONFIGURATION_COLUMNS) + 1):
            nodes = sorted(path for path in centers if len(path) == depth)
            for node in nodes:
                parent = node[:-1]
                parent_y = centers[parent]
                node_y = centers[node]
                elbow_x = depth - 0.5
                tree_ax.plot(
                    [depth - 1, elbow_x, elbow_x, depth],
                    [parent_y, parent_y, node_y, node_y],
                    color="#91a0a8",
                    linewidth=0.7,
                    zorder=1,
                )
                label = f"{node[-1]:g}"
                if depth == len(CONFIGURATION_COLUMNS):
                    tree_ax.plot(
                        [depth, depth + 0.9],
                        [node_y, node_y],
                        color="#e8ecee",
                        linewidth=0.45,
                        zorder=0,
                    )
                tree_ax.scatter(depth, node_y, s=8, color="#40535d", zorder=2)
                tree_ax.text(
                    depth + 0.06,
                    node_y,
                    label,
                    va="center",
                    fontsize=6.5,
                    bbox={"facecolor": "white", "edgecolor": "none", "pad": 0.4},
                    zorder=3,
                )

        tree_ax.set_xlim(-0.1, len(CONFIGURATION_COLUMNS) + 0.9)
        tree_ax.set_xticks(
            np.arange(1, len(CONFIGURATION_COLUMNS) + 1),
            ["participants", "full", "partial", "sigma", "lam"],
        )
        tree_ax.xaxis.tick_top()
        tree_ax.tick_params(axis="x", length=0, labelsize=8)
        tree_ax.tick_params(axis="y", length=0, labelleft=False)
        tree_ax.set_title("Configuration hierarchy", pad=24)
        tree_ax.spines[:].set_visible(False)
        figures[str(parameter)] = fig
    return figures

In [ ]:
recovery_error_data = build_recovery_error_data(sweep, posterior_summary)
mcmc_quality_summary = summarize_mcmc_quality(recovery_error_data)
valid_recovery_error_data = recovery_error_data.loc[
    ~recovery_error_data["mcmc_issue"]
].reset_index(drop=True)
configuration_error_summary = summarize_configuration_errors(valid_recovery_error_data)

display(mcmc_quality_summary)
display(
    pd.Series(
        {
            "all parameter-job observations": len(recovery_error_data),
            "valid parameter-job observations": len(valid_recovery_error_data),
            "stimulus configurations": len(
                recovery_error_data[list(CONFIGURATION_COLUMNS)].drop_duplicates()
            ),
            "minimum valid jobs per configuration": int(
                configuration_error_summary["n_jobs"].min()
            ),
            "maximum valid jobs per configuration": int(
                configuration_error_summary["n_jobs"].max()
            ),
        },
        name="Recovery summary",
    )
)

configuration_error_figures = plot_configuration_error_ranges(
    configuration_error_summary
)
for figure in configuration_error_figures.values():
    plt.close(figure)
for parameter, figure in configuration_error_figures.items():
    display(Markdown(f"### {parameter}"))
    display(figure)

## Per-parameter diagnostics

The following helpers build coverage, rank-uniformity, marginal-density, and per-job recovery panels for one stimulus configuration and parameter.


In [ ]:
def kde_xy(values: np.ndarray | xr.DataArray) -> tuple[np.ndarray, np.ndarray]:
    values = np.asarray(values).reshape(-1)
    values = values[np.isfinite(values)]
    if values.size < 2 or np.isclose(np.std(values), 0.0):
        return np.array([]), np.array([])
    x, y = az.kde(values)[:2]
    mask = np.isfinite(x) & np.isfinite(y)
    return x[mask], y[mask]


def scaled_kde(
    values: np.ndarray | xr.DataArray,
    *,
    x_limits: tuple[float, float],
    height: float = 1.0,
) -> tuple[np.ndarray, np.ndarray]:
    x, y = kde_xy(values)
    if x.size == 0 or y.max() <= 0:
        return np.array([]), np.array([])
    mask = (x >= x_limits[0]) & (x <= x_limits[1])
    if not mask.any():
        return np.array([]), np.array([])
    return x[mask], y[mask] / y.max() * height


def parameter_limits(
    sweep: xr.Dataset, view: xr.Dataset, parameter: str
) -> tuple[float, float]:
    truth = np.asarray(view["truth"].sel(parameter=parameter)).reshape(-1)
    truth = truth[np.isfinite(truth)]
    values = np.concatenate(
        [
            np.asarray(view["posterior"].sel(parameter=parameter)).reshape(-1),
            truth,
            np.asarray(sweep["prior"].sel(parameter=parameter)).reshape(-1),
        ]
    )
    values = values[np.isfinite(values)]
    if values.size == 0:
        return 0.0, 1.0
    lo, hi = az.eti(values, prob=0.99)
    if not np.isfinite(lo) or not np.isfinite(hi) or np.isclose(lo, hi):
        center = values.mean()
        lo, hi = center - 0.5, center + 0.5
    lo = min(lo, truth.min())
    hi = max(hi, truth.max())
    pad = 0.06 * (hi - lo)
    return lo - pad, hi + pad

In [ ]:
def plot_coverage_panel(
    ax: plt.Axes,
    sbc: xr.Dataset,
    *,
    envelope_prob: float = 0.94,
    n_simulations: int = 1000,
    random_seed: int = 0,
) -> None:
    if sbc.sizes.get("job_id", 0) == 0:
        ax.text(
            0.5, 0.5, "no SBC ranks", ha="center", va="center", transform=ax.transAxes
        )
        return
    scores = np.asarray(sbc["coverage_score"]).reshape(-1)
    scores = scores[np.isfinite(scores)]
    if scores.size == 0:
        ax.text(
            0.5,
            0.5,
            "no finite ranks",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        return

    x, empirical_ecdf, lower_band, upper_band = ecdf_pit(
        scores,
        envelope_prob,
        n_simulations=n_simulations,
        rng=np.random.default_rng(random_seed),
    )
    signed_area = 0.5 - np.mean(scores)
    area_se = 1 / np.sqrt(12 * scores.size)
    if abs(signed_area) <= 2 * area_se:
        calibration = "near calibrated"
    else:
        calibration = "overconfident" if signed_area < 0 else "underconfident"

    ax.fill_between(x, lower_band, upper_band, color="#d7ecf7", alpha=0.65, linewidth=0)
    ax.plot([0, 1], [0, 1], color="#1f77b4", linestyle="--", linewidth=1.4)
    ax.plot(x, empirical_ecdf, color="#ff8c2a", linewidth=2.0)
    ax.text(0.02, 0.98, calibration, ha="left", va="top", transform=ax.transAxes)
    ax.set_title("Coverage")
    ax.set_xlabel("Nominal mass q")
    ax.set_ylabel("Empirical inclusion freq.")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.grid(color="#d8d8d8", linewidth=0.8, alpha=0.6)


def plot_ecdf_cdf_panel(
    ax: plt.Axes,
    sbc: xr.Dataset,
    *,
    envelope_prob: float = 0.94,
    n_simulations: int = 1000,
    random_seed: int = 0,
) -> None:
    if sbc.sizes.get("job_id", 0) == 0:
        ax.text(
            0.5, 0.5, "no SBC ranks", ha="center", va="center", transform=ax.transAxes
        )
        return
    pits = np.asarray(sbc["pit"]).reshape(-1)
    pits = pits[np.isfinite(pits) & (pits >= 0) & (pits <= 1)]
    if pits.size < 2:
        ax.text(
            0.5, 0.5, "too few ranks", ha="center", va="center", transform=ax.transAxes
        )
        return

    x, empirical_ecdf, lower_band, upper_band = ecdf_pit(
        pits,
        envelope_prob,
        n_simulations=n_simulations,
        rng=np.random.default_rng(random_seed),
    )
    delta = empirical_ecdf - x
    lower_delta = lower_band - x
    upper_delta = upper_band - x

    ax.fill_between(
        x, lower_delta, upper_delta, color="#bfe6f7", alpha=0.7, linewidth=0
    )
    ax.plot(x, delta, color="#111111", linewidth=1.7)
    ax.axhline(0, color="#777777", linewidth=0.8)
    ax.set_title("ECDF-CDF")
    ax.set_xlabel("Normalized rank threshold z")
    ax.set_ylabel("ECDF - CDF")
    ax.set_xlim(0, 1)
    y_abs = max(
        np.max(np.abs(delta)),
        np.max(np.abs(lower_delta)),
        np.max(np.abs(upper_delta)),
        0.05,
    )
    ax.set_ylim(-1.12 * y_abs, 1.12 * y_abs)
    ax.grid(color="#d8d8d8", linewidth=0.8, alpha=0.6)

In [ ]:
def plot_prior_posterior_marginal_panel(
    ax: plt.Axes,
    sweep: xr.Dataset,
    view: xr.Dataset,
    *,
    parameter: str,
    x_limits: tuple[float, float],
) -> None:
    prior_x, prior_y = kde_xy(sweep["prior"].sel(parameter=parameter))
    posterior_x, posterior_y = kde_xy(view["posterior"].sel(parameter=parameter))

    if prior_x.size:
        mask = (prior_x >= x_limits[0]) & (prior_x <= x_limits[1])
        ax.plot(
            prior_x[mask],
            prior_y[mask],
            color="#2ca02c",
            linewidth=1.8,
            label="Prior pdf",
        )
        ax.fill_between(prior_x[mask], 0, prior_y[mask], color="#2ca02c", alpha=0.2)
    if posterior_x.size:
        mask = (posterior_x >= x_limits[0]) & (posterior_x <= x_limits[1])
        ax.plot(
            posterior_x[mask],
            posterior_y[mask],
            color="#111111",
            linewidth=1.8,
            label="Posterior pdf",
        )
        ax.fill_between(
            posterior_x[mask], 0, posterior_y[mask], color="#111111", alpha=0.15
        )

    ax.set_title("Prior vs Posterior Marginal")
    ax.set_xlabel("parameter value")
    ax.set_ylabel("density")
    ax.set_xlim(*x_limits)
    ax.grid(color="#d8d8d8", linewidth=0.8, alpha=0.6)
    ax.legend(frameon=False)


def plot_parameter_recovery_panel(
    ax: plt.Axes,
    sweep: xr.Dataset,
    view: xr.Dataset,
    *,
    parameter: str,
    x_limits: tuple[float, float],
    ridge_height: float = 0.74,
    r_hat_threshold: float = 1.05,
    min_ess: float = 100.0,
) -> None:
    truth = view["truth"].sel(parameter=parameter).dropna("job_id")
    truth = truth.sortby(truth)
    issue = mcmc_issue(
        view, parameter, r_hat_threshold=r_hat_threshold, min_ess=min_ess
    )

    posterior_color = "#c9e3f2"
    prior_color = "#9dcea0"
    truth_color = "#ff3b35"
    issue_color = "#111111"

    y_prior = truth.sizes.get("job_id", 0) + 0.9
    prior_x, prior_y = scaled_kde(
        sweep["prior"].sel(parameter=parameter), x_limits=x_limits, height=ridge_height
    )
    if prior_x.size:
        ax.fill_between(
            prior_x,
            y_prior,
            y_prior + prior_y,
            color=prior_color,
            alpha=0.68,
            linewidth=0,
        )

    for y_position, job_id in enumerate(truth["job_id"].to_numpy()):
        truth_value = truth.sel(job_id=job_id).item()
        posterior_x, posterior_y = scaled_kde(
            view["posterior"].sel(job_id=job_id, parameter=parameter),
            x_limits=x_limits,
            height=ridge_height,
        )
        if posterior_x.size:
            ax.fill_between(
                posterior_x,
                y_position,
                y_position + posterior_y,
                color=posterior_color,
                alpha=0.72,
                linewidth=0,
            )
        ax.vlines(
            truth_value,
            y_position,
            y_position + ridge_height * 1.08,
            color=truth_color,
            linewidth=1.8,
            zorder=4,
        )
        if issue.sel(job_id=job_id).item():
            ax.scatter(
                [truth_value],
                [y_position + 0.05],
                marker="^",
                color=issue_color,
                s=14,
                zorder=5,
            )

    legend_handles = [
        mpatches.Patch(color=posterior_color, alpha=0.72, label="Posterior pdf"),
        mpatches.Patch(color=prior_color, alpha=0.68, label="Prior pdf"),
        Line2D([], [], color=truth_color, linewidth=1.8, label="True value"),
        Line2D(
            [],
            [],
            marker="^",
            markersize=4,
            linestyle="None",
            color=issue_color,
            label="MCMC issue",
        ),
    ]
    ax.legend(handles=legend_handles, frameon=False, loc="lower right")
    ax.set_title("Parameter Recovery")
    ax.set_xlabel("parameter value")
    ax.set_xlim(*x_limits)
    ax.set_ylim(-0.3, y_prior + ridge_height + 0.35)
    ax.set_yticks([])
    ax.grid(axis="x", color="#d8d8d8", linewidth=0.8, alpha=0.7)
    ax.spines[["top", "right", "left"]].set_visible(False)

In [ ]:
def resolve_diagnostics_figure_size(
    figure_size: tuple[float, float] | str | None,
    view: xr.Dataset,
) -> tuple[float, float]:
    if figure_size is None or figure_size == "auto":
        height = max(8.5, 0.38 * view.sizes.get("job_id", 0) + 5.0)
        return (10.5, height)
    if figure_size == "a4_portrait":
        return A4_PORTRAIT_INCHES
    if figure_size == "a4_landscape":
        return A4_LANDSCAPE_INCHES
    if isinstance(figure_size, str):
        raise ValueError(
            "figure_size must be 'auto', 'a4_portrait', 'a4_landscape', or a (width, height) tuple"
        )
    return figure_size


def plot_parameter_diagnostics(
    sweep: xr.Dataset,
    *,
    stimulus_config: int | str,
    parameter: str,
    envelope_prob: float = 0.94,
    n_simulations: int = 1000,
    random_seed: int = 0,
    ridge_height: float = 0.74,
    r_hat_threshold: float = R_HAT_THRESHOLD,
    min_ess: float = MIN_ESS,
    figure_size: tuple[float, float] | str | None = DIAGNOSTICS_FIGURE_SIZE,
) -> plt.Figure:
    view = stimulus_view(sweep, stimulus_config)
    posterior_values = view["posterior"].sel(parameter=parameter).to_numpy()
    if not np.isfinite(posterior_values).any():
        raise ValueError(
            f"no posterior draws for parameter={parameter!r} and stimulus_config={stimulus_config}"
        )

    label = view["stimulus_label"].isel(job_id=0).item()
    figsize = resolve_diagnostics_figure_size(figure_size, view)
    fig = plt.figure(figsize=figsize, constrained_layout=True)
    grid = fig.add_gridspec(3, 2, width_ratios=[1.0, 1.14])
    axes = {
        "coverage": fig.add_subplot(grid[0, 0]),
        "ecdf": fig.add_subplot(grid[1, 0]),
        "marginal": fig.add_subplot(grid[2, 0]),
        "recovery": fig.add_subplot(grid[:, 1]),
    }

    sbc = parameter_sbc(view, parameter=parameter)
    x_limits = parameter_limits(sweep, view, parameter)
    plot_coverage_panel(
        axes["coverage"],
        sbc,
        envelope_prob=envelope_prob,
        n_simulations=n_simulations,
        random_seed=random_seed,
    )
    plot_ecdf_cdf_panel(
        axes["ecdf"],
        sbc,
        envelope_prob=envelope_prob,
        n_simulations=n_simulations,
        random_seed=random_seed,
    )
    plot_prior_posterior_marginal_panel(
        axes["marginal"],
        sweep,
        view,
        parameter=parameter,
        x_limits=x_limits,
    )
    plot_parameter_recovery_panel(
        axes["recovery"],
        sweep,
        view,
        parameter=parameter,
        x_limits=x_limits,
        ridge_height=ridge_height,
        r_hat_threshold=r_hat_threshold,
        min_ess=min_ess,
    )
    fig.suptitle(f"{label} / {parameter}: per-parameter diagnostics")
    return fig

## Interactive explorer

Choose one stimulus configuration and parameter. Jobs with MCMC issues remain visible and are marked by a small black triangle at the true value.

The controls update one native Matplotlib display in place through Jupyter's `display_id` protocol. This avoids the duplicate `interact` output bug in VS Code without converting the figure into an image widget.


In [ ]:
stimulus_options = [
    (
        str(stimulus_view(sweep, int(config))["stimulus_label"].isel(job_id=0).item()),
        int(config),
    )
    for config in np.unique(sweep["stimulus_config"])
]
parameter_options = [str(parameter) for parameter in sweep["parameter"].to_numpy()]

stimulus_dropdown = widgets.Dropdown(
    options=stimulus_options,
    description="Configuration:",
    layout=widgets.Layout(width="72%"),
)
parameter_dropdown = widgets.Dropdown(
    options=parameter_options,
    description="Parameter:",
    layout=widgets.Layout(width="28%"),
)
parameter_diagnostics_controls = widgets.HBox([stimulus_dropdown, parameter_dropdown])


def build_parameter_diagnostics_figure(
    stimulus_config: int,
    parameter: str,
) -> plt.Figure:
    with plt.ioff():
        figure = plot_parameter_diagnostics(
            sweep,
            stimulus_config=stimulus_config,
            parameter=parameter,
        )
    plt.close(figure)
    return figure


def update_parameter_diagnostics(change: dict[str, object]) -> None:
    changed_stimulus = change["owner"] is stimulus_dropdown
    stimulus_config = (
        int(change["new"]) if changed_stimulus else stimulus_dropdown.value
    )
    parameter = parameter_dropdown.value if changed_stimulus else str(change["new"])
    parameter_diagnostics_display.update(
        build_parameter_diagnostics_figure(stimulus_config, parameter)
    )


display(parameter_diagnostics_controls)
parameter_diagnostics_display = display(
    build_parameter_diagnostics_figure(
        stimulus_dropdown.value,
        parameter_dropdown.value,
    ),
    display_id=True,
)
stimulus_dropdown.observe(update_parameter_diagnostics, names="value")
parameter_dropdown.observe(update_parameter_diagnostics, names="value")